# Aqueous Electrowinning — Full Workflow Notebook
This notebook reproduces the entire modeling suite from Pourbaix to tempered case-hardened product.
Run via `jupyter lab` or `aq-steel --quick` for CLI.

All figures save to `docs/figures/` and reports to `experiments/data/`.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if (ROOT / "models").exists():
    pass
else:
    ROOT = Path.cwd().parent.parent
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from models import *
from models.run_all import main as run_all_main

print("Models imported:", len([x for x in dir() if 'Model' in x]))


## 1. Thermodynamics & Kinetics

In [ ]:
# Pourbaix + kinetics
from models.run_electrochemistry import main as run_ec
run_ec()
from IPython.display import Image
display(Image(filename="docs/figures/pourbaix_fe_h2o.png"))
display(Image(filename="docs/figures/polarization_curves.png"))


## 2. Transport (Nernst-Planck)

In [ ]:
from models.run_transport import main as run_trans
run_trans()
from IPython.display import Image
display(Image(filename="docs/figures/nernst_planck_profiles.png"))


## 3. Pulse-reverse

In [ ]:
from models.run_pulse import main as run_pulse
run_pulse()
from IPython.display import Image
display(Image(filename="docs/figures/pulse_reverse_transient.png"))
display(Image(filename="docs/figures/dc_vs_pulse_comparison.png"))


## 4. Phase I voltammetry + EIS

In [ ]:
from models.run_voltammetry import main as run_volt
from models.run_eis import main as run_eis
run_volt()
run_eis()
display(Image(filename="docs/figures/voltammetry_analysis.png"))
display(Image(filename="docs/figures/eis_nyquist.png"))


## 5. Phase II Hull-cell + gravimetric FE

In [ ]:
from models.run_hull_cell import main as run_hull
run_hull()
display(Image(filename="docs/figures/hull_cell_current_distribution.png"))
display(Image(filename="docs/figures/gravimetric_faradaic_efficiency.png"))


## 6. Phase III co-deposition (anomalous + Guglielmi) + pulse-coupled

In [ ]:
from models.run_co_deposition import main as run_codep
run_codep()
from models.co_deposition import build_phase3_model
model = build_phase3_model(mechanism_fe_ni="hydroxide_suppression")
dc = model.run_at_current(50)
pe = model.run_at_current_pulsed(50,100,duty_cycle=0.5,waveform="pe")
print("DC", dc["alloy_kinetics"])
print("PE", pe["alloy_kinetics"])
display(Image(filename="docs/figures/co_deposition_phase3_combined_hydroxide_suppression.png"))
display(Image(filename="docs/figures/pulse_coupled_co_deposition.png"))


## 7. Mechanical properties (Hall-Petch + SS + dispersion)

In [ ]:
from models.run_mechanical_properties import main as run_mech
run_mech()
display(Image(filename="docs/figures/mechanical_properties_sweep.png"))
display(Image(filename="docs/figures/alloy_vs_mechanical.png"))


## 8. Carburization + carbon potential + tempering

In [ ]:
from models.run_carburization import main as run_carb
from models.run_carbon_potential import main as run_cpot
from models.run_tempering import main as run_temp
run_carb()
run_cpot()
run_temp()

display(Image(filename="docs/figures/carburization_profiles.png"))
display(Image(filename="docs/figures/carbon_potential_map.png"))
display(Image(filename="docs/figures/tempering_curve.png"))
display(Image(filename="docs/figures/case_tempered_hardness.png"))


## 9. Closed-loop + technoeconomics

In [ ]:
from models.run_closed_loop import main as run_cl
from models.run_technoeconomic import main as run_te
from models.run_scenarios import main as run_sc
run_cl()
run_te()
run_sc()
display(Image(filename="docs/figures/scenario_comparison.png"))
display(Image(filename="docs/figures/voltage_breakdown.png"))


## 10. Process flow & master dashboard

In [ ]:
from models.process_flow import generate_process_flow_diagram, generate_detailed_flow_with_composition
from models.run_all import _make_dashboard
generate_process_flow_diagram()
generate_detailed_flow_with_composition()
_make_dashboard()
display(Image(filename="docs/figures/process_flow_diagram.png"))
display(Image(filename="docs/figures/run_all_dashboard.png"))


## Summary JSON

In [ ]:
import json, pathlib
print(json.dumps(json.load(open("experiments/data/master_report.json")) if pathlib.Path("experiments/data/master_report.json").exists() else {"no master":"run_all --quick to generate"}, indent=2)[:5000])
